<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/14-generative-autoregressive-models.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Generative Modeling Foundations and Autoregressive Models** {#generative-modeling-foundations-autoregressive-models}

Generative modeling asks a system to represent how observations are distributed and to create new observations consistent with a condition, context, or learned data distribution. Autoregressive models make this problem tractable by imposing an order and predicting one variable at a time. The same mathematical pattern can generate text tokens, image pixels, audio samples, actions, molecular symbols, or structured records.

This chapter uses scikit-learn's copy of the [UCI Optical Recognition of Handwritten Digits dataset](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B), DOI [10.24432/C50P49](https://doi.org/10.24432/C50P49), CC BY 4.0. Each real 8×8 image is quantized into four grayscale levels and flattened in raster order into 64 discrete tokens. A class-conditional recurrent decoder is trained once and reused for likelihood, exposure-bias, decoding, sampling-control, and evaluation experiments.

![Real UCI-derived digit images quantized into four-level autoregressive token sequences.](assets/dl14-quantized-digits.svg){fig-align="center" width="76%" fig-alt="Ten quantized handwritten digit images, each paired with the beginning of its raster-order pixel-token sequence."}

*Original visualization generated from the chapter dataset. Quantization and raster order match the executable experiment.*

The four-level representation is deliberately modest. It keeps the model CPU-runnable and exposes every probability calculation, but it cannot support conclusions about high-resolution image generation. Generated samples are teaching evidence for mechanisms, not a visual benchmark.

<details>
<summary><strong>PyTorch: establish the discrete image-token experiment</strong></summary>

```python
import copy
import math
import random

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=1414):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_images = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
all_labels = torch.tensor(digits.target, dtype=torch.long)
all_indices = np.arange(len(all_images))
train_idx, holdout_idx = train_test_split(
    all_indices, test_size=0.30, random_state=1414, stratify=digits.target
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=1414,
    stratify=digits.target[holdout_idx],
)
train_images, train_labels = all_images[train_idx], all_labels[train_idx]
val_images, val_labels = all_images[val_idx], all_labels[val_idx]
test_images, test_labels = all_images[test_idx], all_labels[test_idx]

num_levels = 4
bos_token = num_levels


def quantize(images):
    return torch.round(images * (num_levels - 1)).long().flatten(1)


def dequantize(tokens):
    return tokens.float().view(-1, 1, 8, 8) / (num_levels - 1)


def decoder_inputs(tokens):
    bos = torch.full((len(tokens), 1), bos_token, dtype=torch.long)
    return torch.cat([bos, tokens[:, :-1]], dim=1)


train_tokens, val_tokens, test_tokens = quantize(train_images), quantize(val_images), quantize(test_images)
train_decoder_inputs = decoder_inputs(train_tokens)
val_decoder_inputs = decoder_inputs(val_tokens)
test_decoder_inputs = decoder_inputs(test_tokens)

assert all_images.shape == (1797, 1, 8, 8)
assert train_tokens.shape == (1257, 64)
assert train_decoder_inputs[:, 0].eq(bos_token).all()
assert train_tokens.min() == 0 and train_tokens.max() == 3
print({
    "split": (len(train_idx), len(val_idx), len(test_idx)),
    "sequence length": train_tokens.shape[1],
    "pixel vocabulary": list(range(num_levels)),
    "BOS token": bos_token,
    "token frequencies": torch.bincount(train_tokens.flatten(), minlength=num_levels).tolist(),
})
```

</details>

The split occurs before quantization. No generated, validation, or test image enters training. Quantization is a fixed transformation with no fitted statistics, so it cannot leak held-out information.

### **What Does a Generative Model Learn?** {#what-does-a-generative-model-learn}

An unconditional model approximates $p_{\text{data}}(x)$. A conditional model approximates $p_{\text{data}}(x\mid c)$, where $c$ may be a class, prompt, image, speaker, control signal, or previous state. Learning the distribution supports several operations:

- sampling new $x$;
- scoring or comparing observations when likelihood is available;
- completing missing parts through conditional inference;
- compressing data using predicted probabilities;
- learning representations useful for downstream tasks.

These operations are not equivalent. A model can assign useful likelihoods yet produce poor samples under a bad decoder; an implicit model can produce convincing samples without exposing $p(x)$; a conditional generator can ignore $c$ while still matching the marginal $p(x)$. The training objective, model family, and inference algorithm jointly determine what is available.

For this chapter, the desired distribution is

$$
p_{\theta}(x\mid y),\qquad x\in\{0,1,2,3\}^{64},\quad y\in\{0,\ldots,9\}.
$$

$x$ is a quantized image sequence and $y$ is the intended digit class. A valid model should not merely copy training examples: it should assign probability across plausible within-class variations. It must also represent uncertainty. At a background pixel, level 0 may dominate; near a stroke boundary, several levels can be reasonable.

“Learning the data distribution” is always relative to a dataset and representation. The UCI collection omits many handwriting styles, quantization removes intensity detail, and the raster order privileges left-to-right/top-to-bottom dependencies. The learned distribution therefore models this pipeline, not an abstract universal distribution of digits.

### **Generative vs Discriminative Modeling** {#generative-vs-discriminative-modeling}

A discriminative classifier models $p(y\mid x)$ or a decision boundary directly. A generative classifier models $p(x,y)=p(x\mid y)p(y)$ and obtains

$$
p(y\mid x)=\frac{p(x\mid y)p(y)}{\sum_{y'}p(x\mid y')p(y')}.
$$

The generative route can sample $x$, handle some forms of missing data, and incorporate priors. It must model details of $x$ that may be irrelevant to classification, so an incorrect density assumption can reduce predictive accuracy. The discriminative route can focus capacity on the boundary but does not automatically define how to generate an input.

![Discriminative models learn the conditional label boundary, whereas generative models represent observations and labels jointly.](assets/dl14-generative-discriminative.svg){fig-align="center" width="76%" fig-alt="Two panels compare a discriminative mapping from observed x to label y with a generative joint model that samples a class and observation."}

*Original comparison diagram.*

The experiment compares an MLP classifier with a class-conditional naïve categorical pixel model. The latter has an exact likelihood but assumes pixels are conditionally independent given the class.

<details>
<summary><strong>PyTorch: compare discriminative and class-conditional generative classifiers</strong></summary>

```python
class DigitDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU())
        self.head = nn.Linear(64, 10)

    def forward(self, images, return_features=False):
        features = self.encoder(images)
        logits = self.head(features)
        return (logits, features) if return_features else logits


seed_everything(1420)
discriminative_model = DigitDiscriminator()
optimizer = torch.optim.AdamW(discriminative_model.parameters(), lr=3e-3, weight_decay=1e-4)
for _ in range(55):
    optimizer.zero_grad()
    loss = F.cross_entropy(discriminative_model(train_images), train_labels)
    loss.backward()
    optimizer.step()
with torch.no_grad():
    discriminative_accuracy = float((
        discriminative_model(test_images).argmax(1) == test_labels
    ).float().mean())


def fit_naive_categorical(tokens, labels, alpha=1.0):
    counts = torch.full((10, 64, num_levels), alpha)
    for label in range(10):
        class_tokens = tokens[labels == label]
        for level in range(num_levels):
            counts[label, :, level] += (class_tokens == level).sum(dim=0)
    return (counts / counts.sum(dim=-1, keepdim=True)).log()


naive_log_prob = fit_naive_categorical(train_tokens, train_labels, alpha=1.0)
class_log_prior = torch.bincount(train_labels, minlength=10).float().log()
class_log_prior -= torch.logsumexp(class_log_prior, dim=0)


def naive_class_scores(tokens):
    scores = []
    positions = torch.arange(64).unsqueeze(0)
    for label in range(10):
        token_log_prob = naive_log_prob[label][positions, tokens]
        scores.append(token_log_prob.sum(dim=1) + class_log_prior[label])
    return torch.stack(scores, dim=1)


with torch.no_grad():
    generative_accuracy = float((naive_class_scores(test_tokens).argmax(1) == test_labels).float().mean())


def sample_naive_class(label, count, seed=1420):
    generator = torch.Generator().manual_seed(seed + int(label))
    probabilities = naive_log_prob[label].exp()
    samples = torch.multinomial(probabilities, count, replacement=True, generator=generator).T
    return samples


naive_samples = torch.cat([sample_naive_class(label, 2) for label in range(10)], dim=0)
assert naive_samples.shape == (20, 64)
print({"discriminative test accuracy": round(discriminative_accuracy, 3),
       "generative Naive Bayes accuracy": round(generative_accuracy, 3),
       "generated token range": (int(naive_samples.min()), int(naive_samples.max()))})
```

</details>

The comparison is not a verdict that discriminative learning is always better. The generative baseline uses a deliberately weak independence assumption, while the MLP learns interactions. A fair scientific comparison would control parameter count, tune both methods, evaluate missing-data tasks, and quantify both classification and sample quality.

### **Explicit and Implicit Density Models** {#explicit-implicit-density-models}

An **explicit density model** defines or approximates $p_{\theta}(x)$. Autoregressive models and normalizing flows expose tractable likelihoods; latent-variable models often optimize a lower bound because marginalization over latent $z$ is intractable. An **implicit model** defines a sampling process such as $x=G_{\theta}(z)$ but does not provide a tractable normalized density at arbitrary $x$.

![Generative families differ in likelihood access, inference, and sampling path.](assets/dl14-density-taxonomy.svg){fig-align="center" width="76%" fig-alt="Explicit models include tractable likelihood and approximate-bound families, while implicit models provide samples without pointwise density."}

*Original taxonomy. Latent-variable, flow, energy, GAN, and diffusion families are developed in Chapters 15 and 16.*

Likelihood access enables compression and anomaly scores, but likelihood can reward low-level statistics that do not match semantic typicality. Sample-only access allows flexible generators, but evaluation must compare distributions through samples or critics. Neither category guarantees fast sampling: an autoregressive explicit model may require thousands of serial steps, while an implicit feed-forward generator may sample in one pass.

The code contrasts the exact naïve density with a bootstrap-and-jitter sampler. The sampler is intentionally simple: it defines a procedure but no unique tractable probability for an arbitrary output after clipping and random perturbation.

<details>
<summary><strong>Python: contrast pointwise likelihood with sample-only access</strong></summary>

```python
def naive_sequence_log_prob(tokens, labels):
    positions = torch.arange(64).unsqueeze(0)
    rows = []
    for row, label in enumerate(labels):
        rows.append(naive_log_prob[int(label)][positions[0], tokens[row]].sum())
    return torch.stack(rows)


def implicit_bootstrap_sample(count, seed=1430):
    generator = torch.Generator().manual_seed(seed)
    selected = torch.randint(len(train_images), (count,), generator=generator)
    noise = 0.08 * torch.randn((count, 1, 8, 8), generator=generator)
    return (train_images[selected] + noise).clamp(0.0, 1.0)


explicit_test_nll = float(-naive_sequence_log_prob(test_tokens, test_labels).mean())
implicit_samples = implicit_bootstrap_sample(40)
nearest_distance = torch.cdist(implicit_samples.flatten(1), train_images.flatten(1)).min(dim=1).values

assert math.isfinite(explicit_test_nll)
assert implicit_samples.shape == (40, 1, 8, 8)
print({"explicit mean test NLL": round(explicit_test_nll, 2),
       "implicit samples": len(implicit_samples),
       "implicit mean nearest-train distance": round(float(nearest_distance.mean()), 3),
       "implicit log_prob(x)": "not available"})
```

</details>

A nearest-neighbor distance close to zero would reveal copying, but a large distance does not prove quality. The example's jitter can create novel arrays that are not plausible digits. Generative evaluation needs both fidelity and coverage, and its feature space must match the domain.

### **Maximum Likelihood Estimation** {#maximum-likelihood-estimation}

Maximum likelihood estimation (MLE) chooses parameters that maximize the probability of observed training examples:

$$
\theta^{*}=\arg\max_{\theta}\sum_{n=1}^{N}\log p_{\theta}(x^{(n)}),
\qquad
\mathcal{L}_{\text{NLL}}=-\frac{1}{N}\sum_{n=1}^{N}\log p_{\theta}(x^{(n)}).
$$

For categorical next-token distributions, NLL is cross-entropy between the observed token and predicted probabilities. Minimizing expected NLL is equivalent to minimizing $\mathrm{KL}(p_{\text{data}}\|p_{\theta})$ up to the fixed entropy of the data distribution. This direction strongly penalizes assigning low probability to observed modes, encouraging coverage, but finite data and model misspecification still matter.

The naïve pixel model has a closed-form MLE: normalized counts at each class, position, and level. Without smoothing, unseen validation events receive zero probability and infinite NLL. A Dirichlet/Laplace pseudocount $\alpha$ trades bias for finite support.

<details>
<summary><strong>Python: inspect smoothing and held-out negative log-likelihood</strong></summary>

```python
def categorical_model_nll(alpha):
    log_probability = fit_naive_categorical(train_tokens, train_labels, alpha=alpha)
    positions = torch.arange(64).unsqueeze(0)
    row_nll = []
    for row, label in enumerate(val_labels):
        selected = log_probability[int(label)][positions[0], val_tokens[row]]
        row_nll.append(-selected.sum())
    return float(torch.stack(row_nll).mean())


smoothing_results = {alpha: categorical_model_nll(alpha) for alpha in (0.01, 0.1, 1.0, 5.0)}
uniform_nll = 64 * math.log(num_levels)
best_alpha = min(smoothing_results, key=smoothing_results.get)
assert all(math.isfinite(value) for value in smoothing_results.values())
print({"validation NLL by alpha": {k: round(v, 2) for k, v in smoothing_results.items()},
       "uniform-model NLL": round(uniform_nll, 2), "selected alpha": best_alpha})
```

</details>

Likelihood must be reported in a comparable unit. For images, bits per dimension is $\mathrm{NLL}/(D\log 2)$, where $D$ is the number of modeled scalar dimensions. Comparing models with different quantization, dequantization, tokenization, or preprocessing can be invalid. Training NLL alone is not evidence of generalization; held-out likelihood and sample behavior are both needed.

### **Autoregressive Factorization** {#autoregressive-factorization}

The chain rule factorizes any ordered joint distribution:

$$
p(x_1,\ldots,x_T\mid c)=\prod_{t=1}^{T}p(x_t\mid x_{<t},c),
\qquad
\log p(x\mid c)=\sum_{t=1}^{T}\log p(x_t\mid x_{<t},c).
$$

This identity is exact; approximation enters through the neural conditional distributions and chosen order. The order changes which dependencies are easy to learn. Raster-order images privilege previous rows and columns. Left-to-right text is natural for continuation but inconvenient for bidirectional completion. Channel order matters for color images.

![A 64-token image likelihood decomposed into ordered conditional probabilities.](assets/dl14-autoregressive-factorization.svg){fig-align="center" width="76%" fig-alt="Pixel tokens x1 through x64 form a chain where each conditional probability depends only on earlier tokens."}

*Original chain-rule visualization based on the autoregressive image formulation used in [PixelRNN](https://proceedings.mlr.press/v48/oord16.html).* 

During training, every true prefix is known, so all token losses can be accumulated in one batch. During generation, $x_t$ must be selected before the prefix for $x_{t+1}$ exists. The following recurrent model learns a class-conditional distribution over the 64 quantized pixels.

<details>
<summary><strong>PyTorch: train one class-conditional autoregressive pixel decoder</strong></summary>

```python
class ConditionalPixelRNN(nn.Module):
    def __init__(self, embedding_dim=24, hidden_dim=72):
        super().__init__()
        self.token_embedding = nn.Embedding(num_levels + 1, embedding_dim)
        self.class_embedding = nn.Embedding(10, embedding_dim)
        self.rnn = nn.GRU(embedding_dim * 2, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, num_levels)

    def forward(self, input_tokens, labels):
        token_state = self.token_embedding(input_tokens)
        class_state = self.class_embedding(labels).unsqueeze(1).expand(-1, input_tokens.shape[1], -1)
        hidden, _ = self.rnn(torch.cat([token_state, class_state], dim=-1))
        return self.output(hidden)


def sequence_nll(model, inputs, targets, labels):
    model.eval()
    with torch.no_grad():
        logits = model(inputs, labels)
        per_token = F.cross_entropy(logits.transpose(1, 2), targets, reduction="none")
    return per_token.sum(dim=1)


seed_everything(1440)
autoregressive_model = ConditionalPixelRNN()
optimizer = torch.optim.AdamW(autoregressive_model.parameters(), lr=2e-3, weight_decay=1e-4)
loader = DataLoader(
    TensorDataset(train_decoder_inputs, train_tokens, train_labels),
    batch_size=160, shuffle=True, generator=torch.Generator().manual_seed(1440),
)
best_state, best_validation_nll = copy.deepcopy(autoregressive_model.state_dict()), float("inf")
for _ in range(42):
    autoregressive_model.train()
    for input_batch, target_batch, label_batch in loader:
        optimizer.zero_grad()
        logits = autoregressive_model(input_batch, label_batch)
        loss = F.cross_entropy(logits.transpose(1, 2), target_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(autoregressive_model.parameters(), 1.0)
        optimizer.step()
    validation_nll = float(sequence_nll(
        autoregressive_model, val_decoder_inputs, val_tokens, val_labels
    ).mean())
    if validation_nll < best_validation_nll:
        best_validation_nll = validation_nll
        best_state = copy.deepcopy(autoregressive_model.state_dict())
autoregressive_model.load_state_dict(best_state)

test_ar_nll = float(sequence_nll(
    autoregressive_model, test_decoder_inputs, test_tokens, test_labels
).mean())
assert test_ar_nll < uniform_nll
print({"best validation NLL": round(best_validation_nll, 2),
       "test autoregressive NLL": round(test_ar_nll, 2),
       "test bits/dimension": round(test_ar_nll / (64 * math.log(2)), 3)})
```

</details>

The RNN shares parameters across positions, while the class embedding supplies $c=y$. It still has no explicit 2D inductive bias beyond raster order. Better likelihood does not guarantee visually preferred samples, and a conditional model can exploit class imbalance or ignore the condition; both must be tested separately.

### **Teacher Forcing and Exposure Bias** {#teacher-forcing-exposure-bias}

Teacher forcing supplies the true previous token while training:

$$
\mathcal{L}_{\text{TF}}=-\sum_{t=1}^{T}\log p_{\theta}(x_t^{*}\mid x_{<t}^{*},c).
$$

At inference, the model conditions on its own sampled history $\hat x_{<t}$. A wrong early token can move the model into a prefix region rarely observed during training, and later errors can compound. This train/inference context mismatch is commonly called exposure bias.

![Teacher forcing sees gold histories, while free-running generation feeds model outputs back as later context.](assets/dl14-teacher-forcing.svg){fig-align="center" width="74%" fig-alt="The training row uses gold previous tokens, while the inference row uses sampled tokens and an early error changes later context."}

*Original teaching diagram informed by [Scheduled Sampling](https://papers.nips.cc/paper_files/paper/2015/hash/e995f98d56967d946471af29d7bf99f1-Abstract.html).* 

Scheduled sampling sometimes replaces gold prefixes with generated tokens during training, but it changes the training distribution and is not a universally consistent fix. Sequence-level objectives, imitation learning, denoising, robust conditioning, and better model calibration offer other approaches. The first diagnostic should be to measure sensitivity to prefix perturbations rather than assuming exposure bias explains every poor sample.

<details>
<summary><strong>PyTorch: measure teacher-forced accuracy and prefix-error propagation</strong></summary>

```python
autoregressive_model.eval()
with torch.no_grad():
    teacher_logits = autoregressive_model(test_decoder_inputs, test_labels)
    teacher_token_accuracy = float((teacher_logits.argmax(-1) == test_tokens).float().mean())

    clean_prefix = test_decoder_inputs[:64].clone()
    corrupted_prefix = clean_prefix.clone()
    # Position 9 is the input carrying the true token from target position 8.
    corrupted_prefix[:, 9] = (corrupted_prefix[:, 9] + 1) % num_levels
    clean_logits = autoregressive_model(clean_prefix, test_labels[:64])
    corrupted_logits = autoregressive_model(corrupted_prefix, test_labels[:64])
    clean_log_prob = clean_logits.log_softmax(-1)
    corrupt_log_prob = corrupted_logits.log_softmax(-1)
    propagation_kl = F.kl_div(
        corrupt_log_prob[:, 9:], clean_log_prob[:, 9:].exp(), reduction="none"
    ).sum(-1).mean(dim=0)

assert teacher_token_accuracy > 0.70
assert torch.all(propagation_kl >= -1e-6)
print({"teacher-forced token accuracy": round(teacher_token_accuracy, 3),
       "mean KL immediately after corruption": round(float(propagation_kl[0]), 4),
       "mean KL ten steps later": round(float(propagation_kl[min(10, len(propagation_kl)-1)]), 4)})
```

</details>

Paired pixel accuracy under a free rollout is not a valid primary metric because many different images can represent the same digit. The KL diagnostic instead asks whether one changed history alters later predictive distributions. In language, error accumulation should be separated from ambiguity: a generated continuation can differ from the reference and still be valid.

### **PixelCNN and WaveNet** {#pixelcnn-wavenet}

Autoregressive convolution replaces recurrence with a causal receptive field. PixelCNN predicts image pixels using masked 2D convolutions. A type-A mask excludes the current pixel in the first layer; later type-B masks may include the current hidden location while still excluding future raw pixels. [PixelRNN/PixelCNN](https://proceedings.mlr.press/v48/oord16.html) showed that discrete pixel likelihoods can model complex image dependencies.

WaveNet applies causal 1D convolutions to raw audio. Dilations $1,2,4,\ldots$ expand the receptive field exponentially with depth. For kernel size $k$ and dilation sequence $d_l$, the receptive field is

$$
R=1+(k-1)\sum_l d_l.
$$

Gated activation, residual connections, skip connections, and conditioning help model long waveforms. [WaveNet](https://arxiv.org/abs/1609.03499) remains a canonical demonstration of sample-level autoregression.

![PixelCNN masks future raster positions, while WaveNet dilation expands a causal temporal receptive field.](assets/dl14-causal-convolutions.svg){fig-align="center" width="76%" fig-alt="A raster mask exposes only previous pixels, and a dilated temporal convolution tree reaches distant earlier samples."}

*Original structural diagram based on [PixelRNN](https://proceedings.mlr.press/v48/oord16.html) and [WaveNet](https://arxiv.org/abs/1609.03499).* 

<details>
<summary><strong>PyTorch: construct causal PixelCNN masks and WaveNet receptive fields</strong></summary>

```python
class MaskedConv2d(nn.Conv2d):
    def __init__(self, mask_type, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if mask_type not in {"A", "B"}:
            raise ValueError(mask_type)
        self.register_buffer("mask", torch.ones_like(self.weight))
        center_y, center_x = self.kernel_size[0] // 2, self.kernel_size[1] // 2
        self.mask[:, :, center_y + 1:, :] = 0
        self.mask[:, :, center_y, center_x + 1:] = 0
        if mask_type == "A":
            self.mask[:, :, center_y, center_x] = 0

    def forward(self, inputs):
        return F.conv2d(inputs, self.weight * self.mask, self.bias,
                        self.stride, self.padding, self.dilation, self.groups)


masked_a = MaskedConv2d("A", 1, 4, kernel_size=3, padding=1, bias=False)
masked_b = MaskedConv2d("B", 4, 4, kernel_size=3, padding=1, bias=False)
assert masked_a.mask[0, 0, 1, 1] == 0
assert masked_b.mask[0, 0, 1, 1] == 1
assert masked_a.mask[0, 0, 1, 2] == 0


def receptive_field(kernel_size, dilations):
    return 1 + (kernel_size - 1) * sum(dilations)


dilations = [1, 2, 4, 8, 16]
wave_receptive_field = receptive_field(kernel_size=2, dilations=dilations)
assert wave_receptive_field == 32
print({"PixelCNN A-mask active weights": int(masked_a.mask.sum()),
       "PixelCNN B-mask active weights": int(masked_b.mask.sum()),
       "WaveNet dilations": dilations, "receptive field": wave_receptive_field})
```

</details>

Convolution parallelizes training across positions, but naïve ancestral sampling remains sequential because each new pixel or audio sample changes the next input. Caching and specialized kernels reduce repeated work; multiscale, subscale, or latent approaches trade exact ordering for faster generation. Mask correctness needs unit tests: one future pixel must never influence an earlier output.

### **Autoregressive Decoder Models** {#autoregressive-decoder-models}

A decoder-only Transformer embeds previous tokens, adds positional information, applies causal self-attention and feed-forward blocks, then predicts the next-token distribution. The causal mask sets future attention logits to $-\infty$. Unlike an RNN, teacher-forced training processes all positions in parallel; unlike a masked encoder, each representation may use only its prefix.

![Teacher-forced decoder training predicts all positions in parallel, whereas inference appends one token at a time.](assets/dl14-decoder-training-inference.svg){fig-align="center" width="76%" fig-alt="The training panel processes all masked positions together; the inference panel repeatedly samples and appends tokens while reusing a key-value cache."}

*Original teaching diagram based on the causal decoder in [Attention Is All You Need](https://arxiv.org/abs/1706.03762).* 

At inference, a key-value cache stores attention keys and values from earlier layers. For batch $B$, heads $H$, cached length $T$, and head dimension $d_h$, each layer stores tensors approximately shaped `[B, H, T, d_h]` for both K and V. Cache memory grows linearly with generated length, while attention computation per new token still grows with context length.

The tiny Transformer below is not trained to replace the shared RNN. It verifies the causal contract: changing future input tokens must not change earlier logits.

<details>
<summary><strong>PyTorch: implement and unit-test a causal Transformer decoder</strong></summary>

```python
class TinyCausalDecoder(nn.Module):
    def __init__(self, width=32, heads=4, max_length=64):
        super().__init__()
        self.token_embedding = nn.Embedding(num_levels + 1, width)
        self.class_embedding = nn.Embedding(10, width)
        self.position = nn.Parameter(torch.zeros(1, max_length, width))
        layer = nn.TransformerEncoderLayer(width, heads, dim_feedforward=64,
                                           dropout=0.0, batch_first=True)
        self.blocks = nn.TransformerEncoder(layer, num_layers=2)
        self.output = nn.Linear(width, num_levels)

    def forward(self, input_tokens, labels):
        length = input_tokens.shape[1]
        hidden = self.token_embedding(input_tokens) + self.position[:, :length]
        hidden = hidden + self.class_embedding(labels).unsqueeze(1)
        causal_mask = torch.triu(torch.ones(length, length, dtype=torch.bool), diagonal=1)
        return self.output(self.blocks(hidden, mask=causal_mask))


seed_everything(1460)
causal_decoder = TinyCausalDecoder()
prefix_a = test_decoder_inputs[:3, :20].clone()
prefix_b = prefix_a.clone()
prefix_b[:, 12:] = torch.randint(0, num_levels, prefix_b[:, 12:].shape)
causal_decoder.eval()
with torch.no_grad():
    logits_a = causal_decoder(prefix_a, test_labels[:3])
    logits_b = causal_decoder(prefix_b, test_labels[:3])
    past_difference = float((logits_a[:, :12] - logits_b[:, :12]).abs().max())

parameter_count = sum(p.numel() for p in causal_decoder.parameters())
kv_elements_per_layer = 2 * 3 * 4 * 20 * (32 // 4)
assert past_difference < 1e-6
print({"decoder parameters": parameter_count,
       "future-to-past max difference": past_difference,
       "illustrative K/V elements per layer": kv_elements_per_layer})
```

</details>

Causality tests should run after architecture changes, fused-attention substitutions, packed sequences, and cache updates. Off-by-one errors can leak the target token during training and produce excellent loss with unusable generation. The training input must be shifted so the logit at position $t$ predicts target $x_t$, not the token already visible at that position.

### **Greedy, Beam, and Sampling-Based Generation** {#greedy-beam-sampling-generation}

Training defines conditional probabilities; decoding turns them into a sequence.

- **Greedy decoding** selects $\arg\max_v p(v\mid x_{<t})$ at every step. It is deterministic and cheap but can make an irreversible local mistake.
- **Beam search** retains the top $B$ partial sequences by cumulative log probability. It approximates a high-probability sequence but does not guarantee the global optimum when pruned.
- **Sampling** draws from the conditional distribution. It represents uncertainty and diversity but can enter low-quality tails.

![Greedy, beam, and stochastic decoding explore different paths through the same probability tree.](assets/dl14-decoding-strategies.svg){fig-align="center" width="76%" fig-alt="A probability tree illustrates one greedy path, several beam paths, and probability-weighted sampled paths."}

*Original decoding comparison diagram.*

Fixed-length pixel sequences need no EOS token or length normalization. Text beam search usually requires a length penalty because sums of negative log probabilities favor short sequences. Coverage and repetition penalties are task-specific heuristics, not properties of the probability model.

<details>
<summary><strong>PyTorch: implement greedy, beam, and stochastic pixel decoding</strong></summary>

```python
def next_token_logits(model, generated, labels):
    bos = torch.full((len(labels), 1), bos_token, dtype=torch.long)
    model_input = bos if generated.shape[1] == 0 else torch.cat([bos, generated], dim=1)
    return model(model_input, labels)[:, -1]


@torch.no_grad()
def greedy_decode(model, labels, length=64):
    generated = torch.empty(len(labels), 0, dtype=torch.long)
    for _ in range(length):
        token = next_token_logits(model, generated, labels).argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, token], dim=1)
    return generated


@torch.no_grad()
def sample_decode(model, labels, temperature=1.0, length=64, seed=1470):
    generator = torch.Generator().manual_seed(seed)
    generated = torch.empty(len(labels), 0, dtype=torch.long)
    for _ in range(length):
        logits = next_token_logits(model, generated, labels) / temperature
        token = torch.multinomial(logits.softmax(-1), 1, generator=generator)
        generated = torch.cat([generated, token], dim=1)
    return generated


@torch.no_grad()
def beam_decode_one(model, label, beam_width=4, length=64):
    beams = [(torch.empty(0, dtype=torch.long), 0.0)]
    label_tensor = torch.tensor([label])
    for _ in range(length):
        candidates = []
        for sequence, score in beams:
            logits = next_token_logits(model, sequence.view(1, -1), label_tensor)
            values, indices = logits.log_softmax(-1).topk(beam_width, dim=-1)
            for value, token in zip(values[0], indices[0]):
                candidates.append((torch.cat([sequence, token.view(1)]), score + float(value)))
        beams = sorted(candidates, key=lambda item: item[1], reverse=True)[:beam_width]
    return beams[0]


autoregressive_model.eval()
class_labels = torch.arange(10)
greedy_samples = greedy_decode(autoregressive_model, class_labels)
stochastic_samples = sample_decode(autoregressive_model, class_labels, seed=1471)
beam_sample, beam_log_probability = beam_decode_one(autoregressive_model, label=3)

assert greedy_samples.shape == stochastic_samples.shape == (10, 64)
assert beam_sample.shape == (64,)
print({"greedy unique sequences": int(torch.unique(greedy_samples, dim=0).shape[0]),
       "sampled unique sequences": int(torch.unique(stochastic_samples, dim=0).shape[0]),
       "class-3 beam log probability": round(beam_log_probability, 2)})
```

</details>

Beam search is appropriate for tasks with a narrow set of acceptable outputs and calibrated sequence scores, such as some constrained transduction problems. Open-ended generation often becomes repetitive or generic under likelihood-maximizing decoding. Decoding should be selected against the application utility, not assumed to improve the underlying model.

### **Temperature, Top-k, and Nucleus Sampling** {#temperature-top-k-nucleus-sampling}

Temperature rescales logits $z_v$:

$$
p_T(v)=\frac{\exp(z_v/T)}{\sum_j\exp(z_j/T)}.
$$

$T<1$ sharpens the distribution; $T>1$ flattens it. Top-$k$ sampling keeps only the $k$ highest-probability tokens. Nucleus or top-$p$ sampling keeps the smallest set whose cumulative probability reaches $p$. The nucleus adapts its size to model uncertainty, unlike a fixed $k$. [Holtzman et al.](https://arxiv.org/abs/1904.09751) introduced nucleus sampling in response to degeneration in open-ended neural text generation.

Truncation is a decoding heuristic. It can remove unreliable tail events, but it also changes the model distribution and can discard valid rare tokens. Small pixel vocabulary makes top-$k$/top-$p$ effects easier to inspect but less representative of language vocabularies containing tens of thousands of tokens.

<details>
<summary><strong>PyTorch: implement temperature, top-k, and top-p sampling</strong></summary>

```python
def filter_logits(logits, top_k=None, top_p=None):
    filtered = logits.clone()
    if top_k is not None:
        threshold = filtered.topk(min(top_k, filtered.shape[-1]), dim=-1).values[:, -1:]
        filtered = filtered.masked_fill(filtered < threshold, -float("inf"))
    if top_p is not None:
        sorted_logits, sorted_indices = filtered.sort(dim=-1, descending=True)
        cumulative = sorted_logits.softmax(-1).cumsum(-1)
        remove = cumulative - sorted_logits.softmax(-1) >= top_p
        sorted_logits = sorted_logits.masked_fill(remove, -float("inf"))
        filtered = torch.full_like(filtered, -float("inf")).scatter(1, sorted_indices, sorted_logits)
    return filtered


@torch.no_grad()
def controlled_sample(model, labels, temperature=1.0, top_k=None, top_p=None, seed=1480):
    generator = torch.Generator().manual_seed(seed)
    generated = torch.empty(len(labels), 0, dtype=torch.long)
    for _ in range(64):
        logits = next_token_logits(model, generated, labels) / temperature
        logits = filter_logits(logits, top_k=top_k, top_p=top_p)
        token = torch.multinomial(logits.softmax(-1), 1, generator=generator)
        generated = torch.cat([generated, token], dim=1)
    return generated


generation_labels = torch.arange(10).repeat_interleave(6)
sampling_configs = {
    "cold_T0.6": {"temperature": 0.6},
    "ancestral_T1.0": {"temperature": 1.0},
    "top_k_2": {"temperature": 1.0, "top_k": 2},
    "top_p_0.85": {"temperature": 1.0, "top_p": 0.85},
}
generated_sets = {
    name: controlled_sample(autoregressive_model, generation_labels, seed=1480 + offset, **config)
    for offset, (name, config) in enumerate(sampling_configs.items())
}

for name, samples in generated_sets.items():
    unique_ratio = torch.unique(samples, dim=0).shape[0] / len(samples)
    mean_pixel_variance = float(dequantize(samples).var(dim=0).mean())
    print({"strategy": name, "unique ratio": round(unique_ratio, 3),
           "mean pixel variance": round(mean_pixel_variance, 4)})

assert all(samples.shape == (60, 64) for samples in generated_sets.values())
```

</details>

Low diversity can mean deterministic confidence or mode collapse; high diversity can mean useful variation or noise. Repetition, entropy, distinct-$n$, self-BLEU, pass rates, and human preferences each capture different aspects in text or code. Sampling hyperparameters are part of the deployed system and must be versioned with the model.

### **Evaluating Generative Models** {#evaluating-generative-models}

No single metric captures a generative distribution. Evaluation should separate:

- **density fit:** held-out NLL, perplexity, or bits per dimension when comparable likelihood exists;
- **fidelity/precision:** whether samples lie on the data manifold;
- **coverage/recall:** whether important data modes are represented;
- **conditional consistency:** whether outputs follow labels, prompts, or controls;
- **novelty and memorization:** whether outputs copy training records;
- **task utility and human judgment:** whether samples satisfy the real use case.

[FID](https://proceedings.neurips.cc/paper/2017/hash/8a1d694707eb0fefe65871369074926d-Abstract.html) compares Gaussian feature statistics, but depends on the feature extractor and sample count. [Generative precision and recall](https://proceedings.neurips.cc/paper_files/paper/2018/hash/f7696a9b362ac5a51c3dc8f098b73923-Abstract.html) separates fidelity from mode coverage. Neither should be computed with an inappropriate domain encoder and presented as universal quality.

The compact audit below uses the chapter's discriminative digit classifier only as a domain-specific probe. It measures class-conditioning accuracy, classifier confidence, diversity, and nearest-training distance for every sampling strategy. These metrics can disagree.

<details>
<summary><strong>PyTorch: audit likelihood, conditional fidelity, diversity, and copying</strong></summary>

```python
def generation_audit(tokens, intended_labels):
    images = dequantize(tokens)
    discriminative_model.eval()
    with torch.no_grad():
        logits, features = discriminative_model(images, return_features=True)
        probabilities = logits.softmax(-1)
        conditional_accuracy = float((probabilities.argmax(1) == intended_labels).float().mean())
        confidence = float(probabilities.max(dim=1).values.mean())
        nearest_train = torch.cdist(images.flatten(1), train_images.flatten(1)).min(dim=1).values
    return {
        "conditional accuracy": conditional_accuracy,
        "classifier confidence": confidence,
        "unique ratio": torch.unique(tokens, dim=0).shape[0] / len(tokens),
        "mean nearest-train distance": float(nearest_train.mean()),
        "pixel diversity": float(images.var(dim=0).mean()),
    }


audit_rows = {name: generation_audit(samples, generation_labels)
              for name, samples in generated_sets.items()}
for name, metrics in audit_rows.items():
    print({name: {key: round(value, 3) for key, value in metrics.items()}})

independent_test_nll = float(-naive_sequence_log_prob(test_tokens, test_labels).mean())
evaluation_summary = {
    "independent categorical bits/dim": independent_test_nll / (64 * math.log(2)),
    "autoregressive bits/dim": test_ar_nll / (64 * math.log(2)),
    "real-test classifier accuracy": discriminative_accuracy,
}
assert evaluation_summary["autoregressive bits/dim"] < 2.0
assert all(0.0 <= row["conditional accuracy"] <= 1.0 for row in audit_rows.values())
print({key: round(value, 3) for key, value in evaluation_summary.items()})
```

</details>

The evaluator was trained on real data, so its confidence can be unreliable on generated artifacts. Nearest distance detects exact or near copying only in raw-pixel geometry. A robust study adds human review, repeated seeds, class/subgroup coverage, train-data extraction tests, and an encoder validated for the domain. Report sample count and confidence intervals for distributional metrics.

### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Generative modeling specifies a distribution or sampling process; autoregressive modeling obtains tractable likelihood by decomposing a joint distribution into ordered next-token conditionals. This yields a distinctive trade-off: teacher-forced training is parallel across known positions, while ancestral generation is inherently sequential.

| Decision | Main alternatives | Benefit | Characteristic limitation |
|---|---|---|---|
| Modeling target | $p(x)$, $p(x\mid c)$, or $p(x,y)$ | sampling, conditioning, or generative classification | dataset and representation define the target |
| Density access | explicit or implicit | likelihood/compression or flexible sampling | tractability does not guarantee perceptual quality |
| Order | raster, temporal, token, channel | exact chain-rule factorization | order changes inductive bias and latency |
| Architecture | RNN, masked CNN, causal Transformer | recurrence, locality, or global context | serial generation and cache cost |
| Training context | teacher forcing or mixed/self-fed histories | stable MLE or robustness to generated prefixes | exposure mismatch or biased training objective |
| Decoding | greedy, beam, ancestral, truncated sampling | determinism, search, or diversity | local errors, degeneration, or tail noise |
| Evaluation | NLL, fidelity, coverage, condition, novelty | complementary evidence | every metric has domain and estimator assumptions |

The UCI experiment connected these decisions without changing data. A class-conditional naïve density exposed the cost of an independence assumption. The recurrent decoder improved held-out likelihood by using prefix dependencies. Prefix corruption measured error propagation. Causal masks were unit-tested. One decoder then supported greedy, beam, temperature, top-$k$, and top-$p$ generation, allowing quality, diversity, and copying indicators to be compared under the same model.

The practical workflow is to state the random variables and conditioning contract; define tokenization and order; verify causal shifts and masks; track held-out likelihood in comparable units; separate the model from the decoder; and evaluate fidelity, coverage, conditioning, novelty, robustness, and use-specific utility. Samples are evidence, not proof, and a single attractive grid cannot establish distribution quality.

Chapter 15 relaxes the purely autoregressive construction. It introduces latent-variable models, variational inference, normalizing flows, and energy-based models, which trade exact factorization for latent structure, invertibility, or unnormalized energies.